# Rightmove Property Address Scraper

This notebook scrapes property addresses from Rightmove search results.

## Instructions:
1. Run each cell in order from top to bottom
2. When prompted, upload your Chrome extension (.zip or .crx file)
3. Upload your outcodes JSON file
4. The scraper will automatically process all outcodes and save results to Excel

## What you'll need:
- Chrome extension file for authentication/session management
- JSON file with outcodes in format: `[{"code":1,"outcode":"AB10"},{"code":2,"outcode":"AB11"}, ...]`

## Step 1: Install Required Packages
This cell installs all necessary libraries. Wait for it to complete before moving to the next step.

In [ ]:
# Install Chrome and ChromeDriver
!apt-get update -qq
!apt-get install -y -qq chromium-chromedriver chromium-browser

# Install Python packages including webdriver-manager for better compatibility
!pip install -q selenium==4.15.2 webdriver-manager openpyxl pandas

# Verify installations
import os
chromium_path = '/usr/bin/chromium-browser'
chromedriver_path = '/usr/bin/chromedriver'

if os.path.exists(chromium_path):
    print(f"✅ Chromium browser found at: {chromium_path}")
else:
    print(f"⚠️  Warning: Chromium not found at expected location")

if os.path.exists(chromedriver_path):
    print(f"✅ ChromeDriver found at: {chromedriver_path}")
else:
    print(f"⚠️  Warning: ChromeDriver not found, will use webdriver-manager")

print("✅ All packages installed successfully!")

## Step 2: Import Libraries

In [ ]:
import json
import time
import os
import zipfile
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from google.colab import files

print("✅ Libraries imported successfully!")

## Step 3: Upload Chrome Extension
Click the 'Choose Files' button and select your Chrome extension file (.zip or .crx)

In [ ]:
print("📤 Please upload your Chrome extension file...")
uploaded_extension = files.upload()

# Get the uploaded filename
extension_filename = list(uploaded_extension.keys())[0]
extension_path = os.path.abspath(extension_filename)

print(f"✅ Extension uploaded: {extension_filename}")
print(f"📁 Saved to: {extension_path}")

## Step 4: Upload Outcodes JSON File
Upload your JSON file containing the outcodes to search

In [ ]:
print("📤 Please upload your outcodes JSON file...")
uploaded_json = files.upload()

# Get the uploaded filename and load the data
json_filename = list(uploaded_json.keys())[0]
with open(json_filename, 'r') as f:
    outcodes_data = json.load(f)

print(f"✅ Outcodes loaded: {len(outcodes_data)} outcodes found")
print(f"📋 Preview: {outcodes_data[:3]}...")

## Step 5: Initialize Browser with Extension
This sets up Chrome with your extension. The browser runs in headless mode (no visible window) which is standard for Colab.

**Note**: If you encounter issues with extension loading, the scraper will continue without it. Make sure you're logged in to Rightmove manually first if needed.

In [ ]:
# Import dependencies at module level
from selenium.webdriver.chrome.service import Service
try:
    from webdriver_manager.chrome import ChromeDriverManager
    from webdriver_manager.core.os_manager import ChromeType
    WEBDRIVER_MANAGER_AVAILABLE = True
except ImportError:
    WEBDRIVER_MANAGER_AVAILABLE = False

def setup_driver(extension_path):
    """Initialize Chrome driver with extension loaded"""
    print("\n" + "="*60)
    print("BROWSER INITIALIZATION STARTING")
    print("="*60 + "\n")
    
    chrome_options = Options()
    
    # Chrome options optimized for Colab - prevent crashes
    chrome_options.add_argument('--headless')  # Use stable headless mode
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--disable-software-rasterizer')
    chrome_options.add_argument('--disable-extensions-except')  # Allow only specific extensions
    chrome_options.add_argument('--disable-background-networking')
    chrome_options.add_argument('--disable-background-timer-throttling')
    chrome_options.add_argument('--disable-backgrounding-occluded-windows')
    chrome_options.add_argument('--disable-breakpad')
    chrome_options.add_argument('--disable-component-extensions-with-background-pages')
    chrome_options.add_argument('--disable-features=TranslateUI,BlinkGenPropertyTrees')
    chrome_options.add_argument('--disable-ipc-flooding-protection')
    chrome_options.add_argument('--disable-renderer-backgrounding')
    chrome_options.add_argument('--enable-features=NetworkService,NetworkServiceInProcess')
    chrome_options.add_argument('--force-color-profile=srgb')
    chrome_options.add_argument('--hide-scrollbars')
    chrome_options.add_argument('--metrics-recording-only')
    chrome_options.add_argument('--mute-audio')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--start-maximized')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    
    # Set binary location
    chrome_options.binary_location = '/usr/bin/chromium-browser'
    
    # Disable images for speed
    prefs = {
        'profile.default_content_setting_values': {
            'images': 2,
            'notifications': 2
        }
    }
    chrome_options.add_experimental_option('prefs', prefs)
    chrome_options.add_experimental_option('excludeSwitches', ['enable-automation', 'enable-logging'])
    
    print(f"✓ Chrome options configured")
    print(f"✓ Binary: {chrome_options.binary_location}\n")
    
    # Extension loading
    if extension_path:
        try:
            extract_path = '/tmp/extension'
            os.makedirs(extract_path, exist_ok=True)
            with zipfile.ZipFile(extension_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
            chrome_options.add_argument(f'--load-extension={extract_path}')
            print(f"✓ Extension loaded from {extract_path}\n")
        except Exception as e:
            print(f"⚠️  Extension failed: {e}\n")
    
    errors = []
    
    # Method 1: System ChromeDriver
    print("→ Method 1: System ChromeDriver")
    try:
        service = Service('/usr/bin/chromedriver')
        service.log_path = '/tmp/chromedriver.log'
        driver = webdriver.Chrome(service=service, options=chrome_options)
        print("✅ SUCCESS with Method 1\n")
        return driver
    except Exception as e:
        msg = f"{type(e).__name__}: {str(e)[:150]}"
        errors.append(("Method 1", msg))
        print(f"✗ {msg}\n")
    
    # Method 2: webdriver-manager
    if WEBDRIVER_MANAGER_AVAILABLE:
        print("→ Method 2: webdriver-manager")
        try:
            service = Service(ChromeDriverManager(chrome_type=ChromeType.CHROMIUM).install())
            service.log_path = '/tmp/chromedriver2.log'
            driver = webdriver.Chrome(service=service, options=chrome_options)
            print("✅ SUCCESS with Method 2\n")
            return driver
        except Exception as e:
            msg = f"{type(e).__name__}: {str(e)[:150]}"
            errors.append(("Method 2", msg))
            print(f"✗ {msg}\n")
    
    # Method 3: Default
    print("→ Method 3: Default initialization")
    try:
        driver = webdriver.Chrome(options=chrome_options)
        print("✅ SUCCESS with Method 3\n")
        return driver
    except Exception as e:
        msg = f"{type(e).__name__}: {str(e)[:150]}"
        errors.append(("Method 3", msg))
        print(f"✗ {msg}\n")
    
    # All failed
    print("="*60)
    print("❌ ALL METHODS FAILED")
    print("="*60)
    for method, err in errors:
        print(f"{method}: {err}")
    
    print("\n🔧 SOLUTION: The issue is likely Chrome crashing in Colab.")
    print("Try these steps:")
    print("1. Runtime → Restart runtime")
    print("2. Run ONLY the installation cell")
    print("3. Wait 30 seconds for system to stabilize")
    print("4. Then run remaining cells")
    print("\nIf still failing, the issue may be:")
    print("- Colab environment limitations")
    print("- Try a different Colab instance")
    print("- Extensions may not work in Colab's restricted environment")
    
    raise Exception("Browser initialization failed - see troubleshooting above")

print("✅ Browser setup function ready!")

## Step 6: Define Scraping Functions

In [ ]:
def build_rightmove_url(code, outcode, index=0):
    """Build Rightmove search URL for given code, outcode and page index"""
    base_url = "https://www.rightmove.co.uk/property-for-sale/find.html"
    
    # Build complete URL with all required parameters
    params = [
        "useLocationIdentifier=true",
        f"locationIdentifier=OUTCODE%5E{code}",
        "radius=0.0",
        "_includeSSTC=on",
        f"index={index}",
        "sortType=2",
        "channel=BUY",
        "transactionType=BUY",
        f"displayLocationIdentifier={outcode}.html",
        "includeSSTC=true"
    ]
    
    url = f"{base_url}?{'&'.join(params)}"
    return url

def extract_addresses_from_page(driver):
    """Extract property addresses from current page"""
    addresses = []
    
    try:
        # Wait for the page to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.TAG_NAME, "a"))
        )
        
        # Find all property cards/listings
        # Look for links that contain property addresses
        property_links = driver.find_elements(By.CSS_SELECTOR, "a[target='_blank'][rel='noopener noreferrer']")
        
        for link in property_links:
            try:
                # Get the address text from the link
                address_text = link.text.strip()
                
                if address_text:
                    # Check if this is a "only postcode found" entry
                    # Look for the parent or nearby elements
                    parent = link.find_element(By.XPATH, "./ancestor::*[1]")
                    parent_html = parent.get_attribute('innerHTML')
                    
                    # Check if "Only postcode found" text is present
                    if '📍 Only postcode found' in parent_html or 'Only postcode found' in parent_html:
                        print(f"  ⏭️  Skipping (only postcode): {address_text[:50]}...")
                        continue
                    
                    # Valid address found
                    addresses.append(address_text)
                    print(f"  ✓ Found: {address_text[:50]}...")
            except Exception as e:
                # Skip problematic elements
                continue
    
    except TimeoutException:
        print("  ⚠️  Page load timeout")
    except Exception as e:
        print(f"  ⚠️  Error extracting addresses: {str(e)}")
    
    return addresses

def has_next_page(driver):
    """Check if there's a next page available"""
    try:
        # Look for next page button or pagination indicator
        next_buttons = driver.find_elements(By.CSS_SELECTOR, "button[data-test='pagination-next']")
        if next_buttons and next_buttons[0].is_enabled():
            return True
        
        # Alternative: check for next page link
        next_links = driver.find_elements(By.LINK_TEXT, "Next")
        if next_links:
            return True
            
        return False
    except:
        return False

def scrape_outcode(driver, code, outcode):
    """Scrape all pages for a given code and outcode"""
    print(f"\n🔍 Scraping outcode: {outcode} (code: {code})")
    all_addresses = []
    page_num = 1
    index = 0
    
    while True:
        print(f"  📄 Page {page_num} (index={index})")
        url = build_rightmove_url(code, outcode, index)
        
        try:
            driver.get(url)
            time.sleep(2)  # Wait for page to load
            
            # Extract addresses from current page
            addresses = extract_addresses_from_page(driver)
            
            if not addresses:
                print(f"  ℹ️  No addresses found on page {page_num}. End of results.")
                break
            
            all_addresses.extend(addresses)
            print(f"  📊 Found {len(addresses)} addresses on this page")
            
            # Check if there's a next page
            if not has_next_page(driver):
                print(f"  ℹ️  No more pages available")
                break
            
            # Move to next page (increment index by 24)
            index += 24
            page_num += 1
            
            # Safety limit to avoid infinite loops
            if page_num > 100:
                print(f"  ⚠️  Reached page limit (100 pages)")
                break
                
        except Exception as e:
            print(f"  ❌ Error on page {page_num}: {str(e)}")
            break
    
    print(f"✅ Completed {outcode}: {len(all_addresses)} total addresses found")
    return all_addresses

print("✅ Scraping functions defined!")

## Step 7: Run the Scraper
This will process all outcodes and collect addresses. This may take several minutes depending on the number of outcodes.

In [ ]:
# Initialize results storage
all_results = []

# Setup the driver with comprehensive error handling
print("\n" + "#"*60)
print("# STARTING BROWSER INITIALIZATION")
print("#"*60)
print("Note: This may take a moment. Watch for progress messages below.\n")

try:
    driver = setup_driver(extension_path)
    print("\n" + "#"*60)
    print("# BROWSER INITIALIZATION COMPLETE")
    print("#"*60 + "\n")
except Exception as e:
    print("\n" + "!"*60)
    print("! BROWSER INITIALIZATION FAILED")
    print("!"*60)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    print("\nPlease follow the troubleshooting steps shown above.")
    raise

try:
    # Process each outcode
    total_outcodes = len(outcodes_data)
    print(f"\n📋 Processing {total_outcodes} outcodes...\n")
    
    for idx, outcode_entry in enumerate(outcodes_data, 1):
        code = outcode_entry.get('code', '')
        outcode = outcode_entry.get('outcode', '')
        
        if not code or not outcode:
            print(f"⚠️  Skipping entry {idx}: missing code or outcode")
            continue
        
        print(f"\n{'='*60}")
        print(f"Progress: {idx}/{total_outcodes} outcodes")
        
        # Scrape addresses for this code/outcode combination
        addresses = scrape_outcode(driver, code, outcode)
        
        # Store results with outcode information
        for address in addresses:
            all_results.append({
                'Code': code,
                'Outcode': outcode,
                'Address': address
            })
        
        # Small delay between outcodes
        time.sleep(1)
    
    print(f"\n{'='*60}")
    print(f"\n�� Scraping completed!")
    print(f"📊 Total addresses collected: {len(all_results)}")
    
finally:
    # Always close the driver
    if 'driver' in locals():
        driver.quit()
        print("\n✅ Browser closed")
    else:
        print("\n⚠️  Browser was not initialized, nothing to close")

## Step 8: Export to Excel
Save all collected addresses to an Excel file and download it

In [ ]:
if all_results:
    # Create DataFrame
    df = pd.DataFrame(all_results)
    
    # Generate filename with timestamp
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_filename = f'rightmove_addresses_{timestamp}.xlsx'
    
    # Save to Excel
    df.to_excel(output_filename, index=False, engine='openpyxl')
    
    print(f"✅ Excel file created: {output_filename}")
    print(f"📊 Total rows: {len(df)}")
    print(f"\n📋 Preview of results:")
    print(df.head(10))
    
    # Download the file
    print(f"\n⬇️  Downloading file...")
    files.download(output_filename)
    print(f"✅ Download started! Check your downloads folder.")
else:
    print("⚠️  No addresses were collected. Nothing to export.")

## Summary

✅ **Done!** Your property addresses have been scraped and exported to Excel.

### What happened:
1. Chrome browser was set up with your extension
2. Each outcode was searched on Rightmove
3. All pages were scraped for each outcode (incrementing by 24)
4. Addresses were extracted from property links
5. Entries with "Only postcode found" were filtered out
6. Results were saved to an Excel file

### Troubleshooting:
- If you got few or no results, the extension might need manual authentication
- Try adjusting the wait times in the scraping functions
- Check that your outcodes JSON format matches the expected structure

### Need to run again?
Simply re-run all cells from the top. You'll be prompted to re-upload files.